In [1]:
import h5py
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np

In [2]:
hsrl_file = '../../data/CAMP2Ex_HSRL/CAMP2EX-HSRL2_P3B_20191001_R0.h5'
out_file = '../data/CAMP2Ex_HSRL_20191001_R0.nc'
f = h5py.File(hsrl_file, 'r')
print(list(f.keys()))
dps = f['DataProducts']
nav = f['Nav_Data']

['000_Readme', 'DataProducts', 'Nav_Data', 'State', 'UserInput', 'header']


In [3]:
gps_time = nav['gps_time'][:,0]
gps_time[804:] += 24
hsrl_xr = xr.Dataset(coords=dict(gps_time=gps_time,
                                 altitude=dps['Altitude'][0,:]))
hsrl_xr['gps_alt'] = (['gps_time'], nav['gps_alt'][:,0])
hsrl_xr['gps_lat'] = (['gps_time'], nav['gps_lat'][:,0])
hsrl_xr['gps_lon'] = (['gps_time'], nav['gps_lon'][:,0])
hsrl_xr['ext532'] = (['gps_time', 'altitude'], dps['532_ext'][:])
hsrl_xr['ext355'] = (['gps_time', 'altitude'], dps['355_ext'][:])
hsrl_xr['ext1064'] = (['gps_time', 'altitude'], dps['1064_ext'][:])
hsrl_xr['Angstrom_532_355'] = (['gps_time', 'altitude'], dps['Angstrom_532_355'][:])
hsrl_xr['Angstrom_Dust'] = (['gps_time', 'altitude'], dps['Angstrom_Dust'][:])
hsrl_xr['Angstrom_Spherical'] = (['gps_time', 'altitude'], dps['Angstrom_Spherical'][:])

lam1 = 0.670
lam2 = 0.532
hsrl_xr['ext670_532_355'] = ((lam1/lam2)**(-hsrl_xr.Angstrom_532_355)) * hsrl_xr.ext532
hsrl_xr['ext670_Dust'] = ((lam1/lam2)**(-hsrl_xr.Angstrom_Dust)) * hsrl_xr.ext532
hsrl_xr['ext670_Spherical'] = ((lam1/lam2)**(-hsrl_xr.Angstrom_Spherical)) * hsrl_xr.ext532

ast_time = 24 + 2 + 39/60 + 30/60/60 # hours from midnight UTC (3 October 2019, 02:39:30)

hsrl_xr.to_netcdf(out_file)